# Experiment 50 - Exp22 TTA/Lambda Boundary Extension

**Building on:** exp49 (OOF cmAP=0.94005), which improved exp22 by slightly overweighting the normal 0s windows and increasing lambda to 0.80.

**Hypothesis:** exp49 still ended on the boundary for lambda and 0s TTA weight. Extend those knobs while staying in the exp22 model family and keeping prior shrink/clipping disabled, since exp49 showed both hurt OOF.

This notebook reuses the exp49 setup/model cells, then runs a focused exp50 search over lambda, 0s/2.5s TTA blend, alpha, power, and a small set of exp22-like ensemble weights.


In [ ]:
import json
from pathlib import Path

nb_dir = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
exp49_path = nb_dir / 'exp49_exp22_public_aware.ipynb'
exp49_nb = json.loads(exp49_path.read_text())

# Reuse exp49 cells 1-9: imports, data loading, helpers, model definitions, and OOF component generation.
for cell_idx in range(1, 10):
    print(f'--- executing exp49 cell {cell_idx} ---')
    src = ''.join(exp49_nb['cells'][cell_idx]['source'])
    exec(compile(src, f'exp49_cell_{cell_idx}', 'exec'), globals())


In [ ]:
# Exp50 focused boundary extension

LAMBDAS_EXT = [0.70, 0.80, 0.90, 1.00, 1.10, 1.20]
TTA_W0S     = [0.50, 0.55, 0.60, 0.65, 0.70]
POWERS_EXT  = [0.90, 1.00, 1.10]
ALPHAS      = [0.05, 0.08, 0.10, 0.12]

ENSEMBLE_GRIDS = [
    (0.05, 0.50, 0.45),  # exp22/exp49 best family
    (0.05, 0.55, 0.40),
    (0.10, 0.50, 0.40),
    (0.10, 0.55, 0.35),
]

best_cmap = 0.0
best_cfg = {}
results = []
prior_cache = {}

def get_prior_pair_exp50(lam):
    if lam not in prior_cache:
        prior_cache[lam] = (
            apply_prior(oof_raw_0, oof_meta_sites, oof_meta_hours, prior_tables, lam,
                        prior_shrink=1.0, prior_logit_clip=None),
            apply_prior(oof_raw_250, oof_meta_sites, oof_meta_hours, prior_tables, lam,
                        prior_shrink=1.0, prior_logit_clip=None),
        )
    return prior_cache[lam]

for lam in LAMBDAS_EXT:
    prior_0, prior_250 = get_prior_pair_exp50(lam)
    for wp, wm, wpr in ENSEMBLE_GRIDS:
        fp_0 = wp * oof_proto_0 + wm * oof_mlp_0 + wpr * prior_0
        fp_250 = wp * oof_proto_250 + wm * oof_mlp_250 + wpr * prior_250
        for tta_w0 in TTA_W0S:
            fp_avg = tta_w0 * fp_0 + (1.0 - tta_w0) * fp_250
            for power in POWERS_EXT:
                for alpha in ALPHAS:
                    probs = postprocess(fp_avg, power=power, base_alpha=alpha)
                    cmap = padded_cmap(Y_FULL_aligned, probs)
                    row = {'lam': lam, 'tta_w0': tta_w0, 'power': power, 'alpha': alpha,
                           'wp': wp, 'wm': wm, 'wpr': wpr, 'cmap': cmap}
                    results.append(row)
                    if cmap > best_cmap:
                        best_cmap = cmap
                        best_cfg = row.copy()

df = pd.DataFrame(results).sort_values('cmap', ascending=False).reset_index(drop=True)
print(f'Best OOF cmAP: {best_cmap:.5f}')
print(f'Best config:   {best_cfg}')
print('\nTop 20 configs:')
print(df.head(20).to_string(index=False))
print('\nBest cmAP by lambda:')
print(df.groupby('lam')['cmap'].max().reset_index().to_string(index=False))
print('\nBest cmAP by tta_w0:')
print(df.groupby('tta_w0')['cmap'].max().reset_index().to_string(index=False))

lam_b = best_cfg['lam']; tta_w0_b = best_cfg['tta_w0']
pow_b = best_cfg['power']; alp_b = best_cfg['alpha']
wp_b = best_cfg['wp']; wm_b = best_cfg['wm']; wpr_b = best_cfg['wpr']
prior_0_f, prior_250_f = get_prior_pair_exp50(lam_b)
fp_0_f = wp_b * oof_proto_0 + wm_b * oof_mlp_0 + wpr_b * prior_0_f
fp_250_f = wp_b * oof_proto_250 + wm_b * oof_mlp_250 + wpr_b * prior_250_f
p_best = postprocess(tta_w0_b * fp_0_f + (1.0 - tta_w0_b) * fp_250_f,
                     power=pow_b, base_alpha=alp_b)
auc_b = macro_auc(Y_FULL_aligned, p_best)
cmap_b = padded_cmap(Y_FULL_aligned, p_best)

print('=' * 60)
print('Experiment 50 - Exp22 TTA/lambda boundary extension')
print(f'Best: lambda={lam_b}, tta_w0={tta_w0_b}, power={pow_b}, alpha={alp_b}')
print(f'Weights: wp={wp_b}, wm={wm_b}, wpr={wpr_b}')
print(f'AUC={auc_b:.5f}  cmAP={cmap_b:.5f}')
print(f'vs exp49 OOF (0.94005): delta cmAP = {cmap_b - 0.94005:+.5f}')
print(f'vs exp22 OOF (0.93894): delta cmAP = {cmap_b - 0.93894:+.5f}')
print(f'Total wall time: {(time.time() - _WALL_START)/60:.1f} min')
